In [ ]:
from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages
from langchain_core.messages import SystemMessage, HumanMessage
from langchain_groq import ChatGroq
from typing_extensions import TypedDict, Annotated
from dotenv import load_dotenv
load_dotenv()

# craete a llm
llm = ChatGroq(
    model="qwen/qwen3.6-27b",
    profile={
        "max_input_tokens":131072
    }
)

# create a state type
class State(TypedDict):
    messages: Annotated[list, add_messages]

# create node (say chatbot)
def chatbot(state: State)->str:
    return {"messages": [llm.invoke(state["messages"])]}

# create graph builder
graph_builder = StateGraph(State)

# bind with nodes
graph_builder.add_node("Chat_Bot_Node", chatbot)

# craete edges
graph_builder.add_edge(START, "Chat_Bot_Node")
graph_builder.add_edge("Chat_Bot_Node", END)

# compile the graph
graph = graph_builder.compile()

In [ ]:
llm.invoke("test")

In [ ]:
# Display the Graph

from IPython.display import Image, display

display(
    Image(
        graph.get_graph().draw_png()
    )
)

In [12]:
res = graph.invoke({
    "messages": "hi"
})
res
# res["messages"][-1].content

{'messages': [HumanMessage(content='hi', additional_kwargs={}, response_metadata={}, id='ec9dfb15-bc84-446c-a92c-dd71e15dac5f'),
  AIMessage(content='\n<think>\nHere\'s a thinking process:\n\n1.  **Analyze User Input:**\n   - User said: "hi"\n   - This is a simple greeting.\n\n2.  **Identify Intent:**\n   - The user is initiating a conversation.\n   - No specific question or task is provided.\n\n3.  **Determine Response Strategy:**\n   - Acknowledge the greeting warmly.\n   - Keep it open-ended to encourage the user to share what they need.\n   - Maintain a friendly and helpful tone.\n\n4.  **Draft Response (Mental):**\n   - "Hi there! How can I help you today?"\n   - "Hello! What\'s on your mind?"\n   - "Hi! I\'m here to help. What can I do for you?"\n\n5.  **Refine Response:**\n   - Keep it concise and natural.\n   - "Hi there! How can I help you today?" works well.\n\n6.  **Final Output Generation:** (matches the refined response)\n   - "Hi there! How can I help you today?"✅\n</thin

In [ ]:
# print(res["messages"][-1].content)
graph

In [ ]:
# Streaming the graph output
for event in graph.stream({"messages": "provide a medium paragraph on 'AIML'"}, stream_mode="updates"):
    print(event)
    for value in event.values():
        print(value["messages"][-1].content)

In [ ]:
# Streaming the graph output
for chunk, meta  in graph.stream({"messages": "provide a medium paragraph on 'AIML'"}, stream_mode="messages"):
    print(chunk.content, flush=True)